# IoT Hourly Temperature Forecasting
Clean EDA and SARIMA modeling on the hourly temperature series.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.statespace.sarimax import SARIMAX

ROOT = Path.cwd().parent
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

from preprocess import load_temperature_series, fill_gaps, clip_outliers, train_validation_split, evaluate_forecast

## 1) Load data

In [ ]:
DATA_PATH = ROOT / 'data' / 'Dataset.csv'
y_raw = load_temperature_series(DATA_PATH)
y_raw.head()

## 2) Clean (fill gaps, clip extremes) and quick summary

In [ ]:
y = fill_gaps(y_raw, method='interpolate')
y = clip_outliers(y, 0.01, 0.99)
y.describe()

## 3) Visualize the cleaned series

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 4))
y.plot(ax=ax, title='Hourly Temperature (cleaned)')
plt.show()

## 4) Stationarity + ACF/PACF on differenced series

In [ ]:
diff1 = y.diff().dropna()
fig, axes = plt.subplots(2, 1, figsize=(10, 6))
plot_acf(diff1, ax=axes[0]); axes[0].set_title('ACF (diff 1)')
plot_pacf(diff1, ax=axes[1]); axes[1].set_title('PACF (diff 1)')
plt.tight_layout(); plt.show()

adf_stat, pvalue, *_ = adfuller(diff1)
print(f'ADF p-value after 1 diff: {pvalue:.4f}')

## 5) Train/validation split and SARIMA fit

In [ ]:
train, val = train_validation_split(y, val_steps=24*7)

model = SARIMAX(
    train,
    order=(1, 1, 1),
    seasonal_order=(1, 0, 1, 24),
    enforce_stationarity=False,
    enforce_invertibility=False,
)
fit = model.fit(disp=False)
fc = fit.get_forecast(steps=len(val))
fc_mean = fc.predicted_mean
conf = fc.conf_int(alpha=0.05)
metrics = evaluate_forecast(val, fc_mean)
metrics

## 6) Forecast vs. actuals

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(train.iloc[-24*7:], label='Train')
plt.plot(val, label='Validation')
plt.plot(fc_mean.index, fc_mean.values, label='Forecast')
plt.fill_between(conf.index, conf.iloc[:,0], conf.iloc[:,1], color='#ff9896', alpha=0.3, label='95% CI')
plt.title('Forecast vs Actuals')
plt.legend()
plt.tight_layout()
plt.show()

print(f"MAE: {metrics['mae']:.3f}, RMSE: {metrics['rmse']:.3f}")

### Next steps
- Tune (p,d,q)(P,D,Q,24) or use pmdarima auto_arima for suggestions.
- Add exogenous variables if available (humidity, pressure, weather forecasts).
- Compare with ETS/Prophet if you want a model bake-off.